# R2 Export - Analysis Data

Exports FantasAI analysis data to Cloudflare R2 for frontend consumption.

## Files Exported

* **breakout_candidates.json** - Weekly breakout candidate predictions
* **sleeper_picks.json** - Undervalued sleeper picks for waiver wire
* **export_players_2026_draft.json** - Draft-ready player roster with 2025 stats & combine metrics
* **gold_weekly_stats.json** - Complete weekly player statistics (583k+ records, all sources) 🆕
* **player_news.json** - Recent player news headlines with URLs (top 5 per player)
* **defense_performance.json** - Defense/ST historical performance (weeks 10-18)
* **defense_predictions.json** - ML-powered Defense/ST fantasy points predictions with confidence intervals

## Target Paths

* `fantasai/analysis/breakout_candidates.json`
* `fantasai/analysis/sleeper_picks.json`
* `fantasai/players/export_players_2026_draft.json`
* `fantasai/stats/gold_weekly_stats.json` 🆕
* `fantasai/analysis/player_news.json`
* `fantasai/analysis/defense_performance.json`
* `fantasai/predictions/defense_predictions.json`

## Source Tables

* `main.fantasai.export_breakout_candidates`
* `main.fantasai.export_sleeper_picks`
* `main.fantasai.draft_ready_roster_2026`
* `main.fantasai.gold_weekly_stats` 🆕
* `main.fantasai.export_player_news`
* `main.fantasai.export_defense_performance`
* `main.fantasai.ml_defense_predictions`

## Schedule

Runs daily after breakout predictions job completes (approx 08:00 UTC)

## Source Data Queries

This notebook exports data from pre-built export tables. Here are the queries that create those tables:

### 1. Breakout Candidates Query

**Source Table:** `main.fantasai.export_breakout_candidates`

**Current Structure:**
```sql
SELECT 
  player_name,
  position,
  team,
  snap_share_delta,
  opportunity_score,
  avg_snap_share,
  week
FROM main.fantasai.export_breakout_candidates
ORDER BY opportunity_score DESC
```

**Column Definitions:**
* `opportunity_score` - ML-generated breakout probability (higher = more likely to break out)
* `snap_share_delta` - Change in snap share % from previous weeks
* `avg_snap_share` - Average snap share over recent weeks

**Created By:** Breakout Predictions - Weekly Production Run notebook (runs Tuesday 10 AM ET)

---

### 2. Sleeper Picks Query

**Source Table:** `main.fantasai.export_sleeper_picks`

**Current Structure:**
```sql
SELECT 
  player_name,
  position,
  team,
  ownership_pct,      -- ⚠️ ISSUE: Currently 0 for all records
  projected_pts,
  value_score,
  reason
FROM main.fantasai.export_sleeper_picks
ORDER BY value_score DESC
```

**Column Definitions:**
* `ownership_pct` - Sleeper platform ownership percentage (⚠️ needs fix - see below)
* `projected_pts` - Projected fantasy points
* `value_score` - Combined metric of upside/opportunity
* `reason` - Text explanation for the pick

**Known Issue:** All `ownership_pct` values are 0. Table needs rebuild with proper JOIN:

```sql
CREATE OR REPLACE TABLE main.fantasai.export_sleeper_picks AS
SELECT 
    p.player_name,
    p.position,
    p.team,
    COALESCE(o.ownership_pct, 0) as ownership_pct,  -- JOIN to bronze source
    p.projected_pts,
    p.value_score,
    p.reason
FROM main.fantasai.player_projections p
LEFT JOIN main.fantasai.bronze_sleeper_ownership o
    ON p.player_name = o.player_name
WHERE p.value_score > 20  -- Sleeper threshold
ORDER BY value_score DESC
LIMIT 30;
```

---

### Data Quality Checks

**Before exporting, always validate:**
```sql
-- Check for suspicious zero values
SELECT 
  COUNT(*) as total,
  SUM(CASE WHEN ownership_pct = 0 THEN 1 ELSE 0 END) as zero_ownership
FROM main.fantasai.export_sleeper_picks;
```

In [0]:
import json
import boto3
from datetime import datetime
from pyspark.sql import functions as F

# Configuration
CATALOG = "main"
SCHEMA = "fantasai"

# R2 Configuration (using boto3 S3-compatible API)
# Secrets should be stored in Databricks secrets
R2_ACCESS_KEY_ID = dbutils.secrets.get(scope="r2_credentials", key="r2_access_key_id")
R2_SECRET_ACCESS_KEY = dbutils.secrets.get(scope="r2_credentials", key="r2_secret_access_key")
R2_ENDPOINT_URL = dbutils.secrets.get(scope="r2_credentials", key="r2_endpoint_url")
R2_BUCKET_NAME = dbutils.secrets.get(scope="r2_credentials", key="r2_bucket_name")

print(f"✓ Catalog: {CATALOG}")
print(f"✓ Schema: {SCHEMA}")
print(f"✓ R2 Bucket: {R2_BUCKET_NAME}")
print(f"✓ Current time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

In [0]:
# Initialize boto3 S3 client for R2
s3_client = boto3.client(
    's3',
    endpoint_url=R2_ENDPOINT_URL,
    aws_access_key_id=R2_ACCESS_KEY_ID,
    aws_secret_access_key=R2_SECRET_ACCESS_KEY,
    region_name='auto'  # R2 uses 'auto' for region
)

print("✓ R2 client initialized")

In [0]:
print("="*70)
print("EXPORTING BREAKOUT CANDIDATES")
print("="*70)

# Query breakout candidates table
breakout_df = spark.table(f"{CATALOG}.{SCHEMA}.export_breakout_candidates")

record_count = breakout_df.count()
print(f"\nRecords to export: {record_count}")

if record_count > 0:
    # Convert to JSON
    breakout_data = breakout_df.toPandas().to_dict(orient='records')
    
    # Add metadata
    export_payload = {
        "data": breakout_data,
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "record_count": record_count,
            "source_table": "main.fantasai.export_breakout_candidates"
        }
    }
    
    # Convert to JSON string
    json_string = json.dumps(export_payload, default=str, indent=2)
    
    # Upload to R2
    s3_client.put_object(
        Bucket=R2_BUCKET_NAME,
        Key='analysis/breakout_candidates.json',
        Body=json_string.encode('utf-8'),
        ContentType='application/json',
        CacheControl='public, max-age=3600'
    )
    
    print(f"\n✓ Uploaded to R2: fantasai/analysis/breakout_candidates.json")
    print(f"✓ File size: {len(json_string)} bytes")
    print(f"\nSample record:")
    print(json.dumps(breakout_data[0], indent=2, default=str))
else:
    print("⚠️ No records to export")

In [0]:
print("\n" + "="*70)
print("EXPORTING SLEEPER PICKS")
print("="*70)

# Query sleeper picks table
sleeper_df = spark.table(f"{CATALOG}.{SCHEMA}.export_sleeper_picks")

record_count = sleeper_df.count()
print(f"\nRecords to export: {record_count}")

if record_count > 0:
    # ========================================
    # DATA QUALITY VALIDATION
    # ========================================
    pandas_df = sleeper_df.toPandas()
    
    # Check 1: Team data quality
    unk_team_count = pandas_df['team'].isna().sum() + (pandas_df['team'] == 'UNK').sum()
    unk_team_pct = (unk_team_count / record_count) * 100
    
    print(f"\n📊 Data Quality Checks:")
    print(f"   Team data: {unk_team_count}/{record_count} UNK or NULL ({unk_team_pct:.1f}%)")
    
    if unk_team_pct > 50:
        error_msg = f"⚠️ VALIDATION FAILED: {unk_team_pct:.1f}% of records have UNK/NULL teams (threshold: >50%)"
        print(f"\n{error_msg}")
        raise ValueError(error_msg)
    elif unk_team_pct > 10:
        warning_msg = f"⚠️ WARNING: {unk_team_pct:.1f}% of records have UNK/NULL teams (threshold: >10%)"
        print(f"   {warning_msg}")
    else:
        print(f"   ✓ Team data quality: PASS")
    
    # Check 2: Critical fields present
    required_fields = ['player_name', 'position', 'ownership_pct', 'value_score']
    missing_fields = []
    for field in required_fields:
        if field not in pandas_df.columns:
            missing_fields.append(field)
        elif pandas_df[field].isna().all():
            missing_fields.append(f"{field} (all NULL)")
    
    if missing_fields:
        error_msg = f"⚠️ VALIDATION FAILED: Missing or empty required fields: {', '.join(missing_fields)}"
        print(f"\n{error_msg}")
        raise ValueError(error_msg)
    else:
        print(f"   ✓ Required fields: PASS")
    
    # Check 3: Ownership data quality
    zero_ownership = (pandas_df['ownership_pct'] == 0).sum()
    zero_ownership_pct = (zero_ownership / record_count) * 100
    print(f"   Ownership data: {zero_ownership}/{record_count} with 0% ownership ({zero_ownership_pct:.1f}%)")
    
    if zero_ownership_pct > 80:
        error_msg = f"⚠️ VALIDATION FAILED: {zero_ownership_pct:.1f}% of records have 0% ownership (threshold: >80%)"
        print(f"\n{error_msg}")
        raise ValueError(error_msg)
    else:
        print(f"   ✓ Ownership data quality: PASS")
    
    print(f"\n✓ All validation checks passed!")
    
    # ========================================
    # EXPORT TO R2
    # ========================================
    
    # Convert to JSON
    sleeper_data = pandas_df.to_dict(orient='records')
    
    # Add metadata
    export_payload = {
        "data": sleeper_data,
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "record_count": record_count,
            "source_table": "main.fantasai.export_sleeper_picks",
            "data_quality": {
                "unk_team_pct": round(unk_team_pct, 2),
                "zero_ownership_pct": round(zero_ownership_pct, 2),
                "validation_passed": True
            }
        }
    }
    
    # Convert to JSON string
    json_string = json.dumps(export_payload, default=str, indent=2)
    
    # Upload to R2
    s3_client.put_object(
        Bucket=R2_BUCKET_NAME,
        Key='analysis/sleeper_picks.json',
        Body=json_string.encode('utf-8'),
        ContentType='application/json',
        CacheControl='public, max-age=3600'
    )
    
    print(f"\n✓ Uploaded to R2: fantasai/analysis/sleeper_picks.json")
    print(f"✓ File size: {len(json_string):,} bytes")
    print(f"\nSample record:")
    print(json.dumps(sleeper_data[0], indent=2, default=str))
else:
    print("⚠️ No records to export")

In [0]:
print("\n" + "="*70)
print("EXPORTING PLAYER ROSTER (DRAFT 2026)")
print("="*70)

# Query draft-ready roster table
roster_df = spark.table(f"{CATALOG}.{SCHEMA}.draft_ready_roster_2026")

record_count = roster_df.count()
print(f"\nRecords to export: {record_count}")

if record_count > 0:
    # Convert to JSON
    import math
    
    roster_pandas = roster_df.toPandas()
    roster_data = roster_pandas.to_dict(orient='records')
    
    # Recursively replace NaN with None in the data structure
    def clean_nan(obj):
        if isinstance(obj, dict):
            return {k: clean_nan(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [clean_nan(item) for item in obj]
        elif isinstance(obj, float) and math.isnan(obj):
            return None
        else:
            return obj
    
    roster_data_cleaned = clean_nan(roster_data)
    
    # Add metadata
    export_payload = {
        "data": roster_data_cleaned,
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "record_count": record_count,
            "source_table": "main.fantasai.draft_ready_roster_2026",
            "season": 2026
        }
    }
    
    # Convert to JSON string
    json_string = json.dumps(export_payload, default=str, indent=2)
    
    # Upload to R2
    s3_client.put_object(
        Bucket=R2_BUCKET_NAME,
        Key='fantasai/players/export_players_2026_draft.json',
        Body=json_string.encode('utf-8'),
        ContentType='application/json',
        CacheControl='public, max-age=3600'
    )
    
    print(f"\n✓ Uploaded to R2: fantasai/players/export_players_2026_draft.json")
    print(f"✓ File size: {len(json_string):,} bytes")
    print(f"\nSample record:")
    print(json.dumps(roster_data[0], indent=2))
else:
    print("⚠️ No records to export")

In [0]:
print("\n" + "="*70)
print("EXPORTING GOLD WEEKLY STATS")
print("="*70)

# Query gold weekly stats table
weekly_stats_df = spark.table(f"{CATALOG}.{SCHEMA}.gold_weekly_stats")

record_count = weekly_stats_df.count()
print(f"\nRecords to export: {record_count:,}")

if record_count > 0:
    # Convert to JSON
    import math
    
    weekly_stats_pandas = weekly_stats_df.toPandas()
    weekly_stats_data = weekly_stats_pandas.to_dict(orient='records')
    
    # Recursively replace NaN with None and convert timestamps to strings
    import pandas as pd
    
    def clean_nan(obj):
        if isinstance(obj, dict):
            return {k: clean_nan(v) for k, v in obj.items()}
        elif isinstance(obj, list):
            return [clean_nan(item) for item in obj]
        elif isinstance(obj, float) and math.isnan(obj):
            return None
        elif pd.isna(obj):  # Handles NaT, None, NaN
            return None
        elif isinstance(obj, pd.Timestamp):
            return obj.isoformat()
        else:
            return obj
    
    weekly_stats_data_cleaned = clean_nan(weekly_stats_data)
    
    # Get summary stats by source
    source_summary = weekly_stats_pandas.groupby('source').agg({
        'master_player_id': 'count',
        'season': ['min', 'max'],
        'week': ['min', 'max']
    }).to_dict()
    
    # Add metadata
    export_payload = {
        "data": weekly_stats_data_cleaned,
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "record_count": record_count,
            "source_table": "main.fantasai.gold_weekly_stats",
            "sources": list(weekly_stats_pandas['source'].unique()),
            "seasons": {
                "min": int(weekly_stats_pandas['season'].min()),
                "max": int(weekly_stats_pandas['season'].max())
            },
            "weeks": {
                "min": int(weekly_stats_pandas['week'].min()),
                "max": int(weekly_stats_pandas['week'].max())
            },
            "unique_players": int(weekly_stats_pandas['master_player_id'].nunique()),
            "data_quality": {
                "api_sports_records": int(weekly_stats_pandas[weekly_stats_pandas['source'] == 'api_sports'].shape[0]),
                "sleeper_records": int(weekly_stats_pandas[weekly_stats_pandas['source'] == 'sleeper'].shape[0]),
                "week_18_2025_included": bool(((weekly_stats_pandas['season'] == 2025) & (weekly_stats_pandas['week'] == 18)).any())
            }
        }
    }
    
    # Convert to JSON string
    json_string = json.dumps(export_payload, default=str, indent=2)
    
    # Upload to R2
    s3_client.put_object(
        Bucket=R2_BUCKET_NAME,
        Key='fantasai/stats/gold_weekly_stats.json',
        Body=json_string.encode('utf-8'),
        ContentType='application/json',
        CacheControl='public, max-age=3600'
    )
    
    print(f"\n✓ Uploaded to R2: fantasai/stats/gold_weekly_stats.json")
    print(f"✓ File size: {len(json_string):,} bytes ({len(json_string) / 1024 / 1024:.2f} MB)")
    
    print(f"\n📊 Data Summary by Source:")
    for source in weekly_stats_pandas['source'].unique():
        source_df = weekly_stats_pandas[weekly_stats_pandas['source'] == source]
        print(f"  • {source}: {len(source_df):,} records")
        print(f"    - Players: {source_df['master_player_id'].nunique():,}")
        print(f"    - Seasons: {source_df['season'].min()}-{source_df['season'].max()}")
        print(f"    - Weeks: {source_df['week'].min()}-{source_df['week'].max()}")
    
    print(f"\n✅ Total: {record_count:,} weekly stats records exported")
else:
    print("⚠️ No records to export")

In [0]:
print("\n" + "="*70)
print("EXPORTING PLAYER NEWS")
print("="*70)

# Query player news table
news_df = spark.table(f"{CATALOG}.{SCHEMA}.export_player_news")

record_count = news_df.count()
print(f"\nRecords to export: {record_count}")

if record_count > 0:
    # Convert to JSON (convert timestamp to string first)
    news_pandas = news_df.toPandas()
    news_pandas['published_at'] = news_pandas['published_at'].dt.strftime('%Y-%m-%d %H:%M:%S')
    news_data = news_pandas.to_dict(orient='records')
    
    # Add metadata
    export_payload = {
        "data": news_data,
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "record_count": record_count,
            "total_players": news_df.select("player_id").distinct().count(),
            "max_articles_per_player": 5,
            "data_retention_days": 60,
            "source_table": "main.fantasai.export_player_news"
        }
    }
    
    # Convert to JSON string
    json_string = json.dumps(export_payload, default=str, indent=2)
    
    # Upload to R2
    s3_client.put_object(
        Bucket=R2_BUCKET_NAME,
        Key='analysis/player_news.json',
        Body=json_string.encode('utf-8'),
        ContentType='application/json',
        CacheControl='public, max-age=3600'
    )
    
    print(f"\n✓ Uploaded to R2: fantasai/analysis/player_news.json")
    print(f"✓ File size: {len(json_string)} bytes")
    print(f"\nSample record:")
    print(json.dumps(news_data[0], indent=2, default=str))
else:
    print("⚠️ No records to export")

In [0]:
print("\n" + "="*70)
print("EXPORTING DEFENSE PERFORMANCE")
print("="*70)

# Query defense performance table
defense_df = spark.table(f"{CATALOG}.{SCHEMA}.export_defense_performance")

record_count = defense_df.count()
print(f"\nRecords to export: {record_count}")

if record_count > 0:
    # Convert to JSON
    defense_pandas = defense_df.toPandas()
    defense_data = defense_pandas.to_dict(orient='records')
    
    # Get latest week stats for metadata
    latest_week = defense_pandas['week'].max()
    teams_count = defense_pandas['team'].nunique()
    
    # Add metadata
    export_payload = {
        "data": defense_data,
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "record_count": record_count,
            "unique_teams": teams_count,
            "latest_week": int(latest_week),
            "weeks_included": sorted(defense_pandas['week'].unique().tolist()),
            "source_table": "main.fantasai.export_defense_performance",
            "note": "Historical performance only. ML projections coming soon."
        }
    }
    
    # Convert to JSON string
    json_string = json.dumps(export_payload, default=str, indent=2)
    
    # Upload to R2
    s3_client.put_object(
        Bucket=R2_BUCKET_NAME,
        Key='analysis/defense_performance.json',
        Body=json_string.encode('utf-8'),
        ContentType='application/json',
        CacheControl='public, max-age=3600'
    )
    
    print(f"\n✓ Uploaded to R2: fantasai/analysis/defense_performance.json")
    print(f"✓ File size: {len(json_string):,} bytes")
    print(f"\n📊 Top 5 Defenses (by avg last 4 weeks):")
    
    # Show top 5
    top_defense = defense_pandas[defense_pandas['week'] == latest_week].nlargest(5, 'avg_last_4_weeks')
    for idx, row in top_defense.iterrows():
        print(f"   {row['team_name']} ({row['team']}): {row['avg_last_4_weeks']:.2f} avg pts")
else:
    print("⚠️ No records to export")

In [0]:
print("\n" + "="*70)
print("EXPORTING DEFENSE PREDICTIONS")
print("="*70)

# Query defense predictions table
defense_pred_df = spark.table(f"{CATALOG}.{SCHEMA}.ml_defense_predictions")

record_count = defense_pred_df.count()
print(f"\nRecords to export: {record_count}")

if record_count > 0:
    # Convert to JSON
    defense_pred_pandas = defense_pred_df.toPandas()
    
    # Format for frontend consumption
    defense_pred_data = defense_pred_pandas[[
        'team', 
        'season', 
        'predicted_week',
        'predicted_points',
        'predicted_lower_80',
        'predicted_upper_80',
        'confidence_width',
        'last_week_actual_points',
        'rolling_5g_fantasy_pts',
        'momentum_score',
        'prediction_date',
        'model_version'
    ]].to_dict(orient='records')
    
    # Get stats for metadata
    avg_prediction = defense_pred_pandas['predicted_points'].mean()
    max_prediction = defense_pred_pandas['predicted_points'].max()
    top_team = defense_pred_pandas.loc[defense_pred_pandas['predicted_points'].idxmax(), 'team']
    
    # Add metadata
    export_payload = {
        "data": defense_pred_data,
        "metadata": {
            "generated_at": datetime.now().isoformat(),
            "record_count": record_count,
            "predicted_week": int(defense_pred_pandas['predicted_week'].iloc[0]),
            "season": int(defense_pred_pandas['season'].iloc[0]),
            "model_version": defense_pred_pandas['model_version'].iloc[0],
            "prediction_date": defense_pred_pandas['prediction_date'].iloc[0],
            "avg_predicted_points": float(avg_prediction),
            "max_predicted_points": float(max_prediction),
            "top_defense": top_team,
            "source_table": "main.fantasai.ml_defense_predictions"
        }
    }
    
    # Convert to JSON string
    json_string = json.dumps(export_payload, default=str, indent=2)
    
    # Upload to R2
    s3_client.put_object(
        Bucket=R2_BUCKET_NAME,
        Key='predictions/defense_predictions.json',
        Body=json_string.encode('utf-8'),
        ContentType='application/json',
        CacheControl='max-age=3600'  # Cache for 1 hour
    )
    
    print(f"\n✅ Uploaded {record_count} defense predictions")
    print(f"   Top defense: {top_team} ({max_prediction:.1f} predicted points)")
    print(f"   Avg prediction: {avg_prediction:.1f} points")
    print(f"   Predicted week: {defense_pred_pandas['predicted_week'].iloc[0]}")
    print("   Destination: predictions/defense_predictions.json")
else:
    print("⚠️ No defense predictions found - skipping export")

In [0]:
print("\n" + "="*70)
print("EXPORT SUMMARY")
print("="*70)

print("\n✅ Export Complete!")
print(f"\nFiles exported to R2 bucket '{R2_BUCKET_NAME}':")
print("  • analysis/breakout_candidates.json")
print("  • analysis/sleeper_picks.json")
print("  • players/export_players_2026_draft.json")
print("  • stats/gold_weekly_stats.json 🆕")
print("  • analysis/player_news.json")
print("  • analysis/defense_performance.json")
print("  • predictions/defense_predictions.json")
print(f"\nGenerated at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S UTC')}")
print("\nBackend API endpoints:")
print("  • /api/v1/breakout-candidates")
print("  • /api/v1/sleeper-picks")
print("  • /api/v1/players/draft-2026")
print("  • /api/v1/stats/weekly 🆕")
print("  • /api/v1/player-news")
print("  • /api/v1/defense-performance")
print("  • /api/v1/defense-predictions")